# Classify difference between frog and cats


In [1]:
import torch

In [2]:
import random
import numpy as np
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import torchvision
from torchvision import transforms
from PIL import Image

In [3]:
# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


### Transforms

In [6]:
img_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### Load Images from CIFAR10

In [13]:
from torch.utils.data import Dataset
import torchvision

class BinaryCatFrogCIFAR10(Dataset):
    """
    0 -> cat
    1 -> frog
    """
    def __init__(self, root="./data", train=True, transform=None, download=True):
        self.base = torchvision.datasets.CIFAR10(
            root=root,
            train=train,
            download=download
        )
        self.transform = transform

        classes = self.base.classes

        self.cat_idx = classes.index("cat")
        self.frog_idx = classes.index("frog")

        self.samples = []
        for i, target in enumerate(self.base.targets):
            if target == self.cat_idx:
                self.samples.append((i, 0))
            elif target == self.frog_idx:
                self.samples.append((i, 1))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        base_idx, label = self.samples[idx]
        image, _ = self.base[base_idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [14]:
full_train_dataset = BinaryCatFrogCIFAR10(
    root="./data",
    train=True,
    transform=img_transforms,
    download=True
)

In [15]:
test_data = BinaryCatFrogCIFAR10(
    root="./data",
    train=False,
    transform=img_transforms,
    download=True
)

In [16]:
# Split train into train/val
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_data, val_data = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

### Create DataLoader

In [17]:
batch_size=64
train_data_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size)
val_data_loader  = torch.utils.data.DataLoader(val_data, batch_size=batch_size)
test_data_loader  = torch.utils.data.DataLoader(test_data, batch_size=batch_size)

### Create Model

SimpleNet is a model of three Linear layers and ReLu activations between them.

**Note**! No need for softmax() in the forward(), because crossEntropy loss add it. However, it is needed in the training function during the validation phase.